# Notebook 2 - Data Type Handling

In this notebook I learn how to identify the data type of every column, how to convert one type into another (`astype()`, `to_numeric()`, `to_datetime()`), how to handle values that cannot be converted, and how to detect columns that have the wrong type. At the end I fix the data types of the Telco dataset.

**Dataset:** Telco Customer Churn (7043 rows, 21 columns).

**Note:** The Telco dataset has no date column, so for the date/time topic I use a small demo table that I created myself. I say clearly where I use demo data.

**Rule:** `df` (raw data) is never changed. I do all the work on copies.

In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")   # raw data, never modified
print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Problems from Sprint 4 Related to Data Types

| # | Problem | Why it exists | Planned fix |
|---|---|---|---|
| 1 | `TotalCharges` is text, but it is a number | 11 cells have a blank space, so pandas reads the column as text | `pd.to_numeric(errors="coerce")` |
| 2 | `SeniorCitizen` is 0/1, but other yes/no columns use "Yes"/"No" | the columns follow different export rules | convert all yes/no features to one type (boolean) |
| 3 | Yes/No and other labels are stored as plain text | pandas reads every text column as `object` | convert to `bool` or `category` |
| 4 | There is no date column | the dataset only has `tenure` in months | use a demo table to practise `to_datetime()` |

## 2. Identifying Data Types

**Concept:** The data type (dtype) tells pandas what kind of values a column holds, and so which operations are allowed on it. `int64` and `float64` are numbers, `object` (or `str`) is text, `bool` is True/False, `datetime64` is a date and `category` is a fixed set of labels.

**Examples:**
- Simple: the column `[10, 20, 30]` is `int64`, and `["a", "b"]` is `object`.
- Real-world: a phone number is stored as a number but it is really an identifier.
- Business: `TotalCharges` is a money amount but pandas reads it as text.
- AI/ML: a model needs to know which columns are numeric and which are categorical to choose the right preprocessing.

**Implementation:** I look at the dtype, the number of unique values, the missing values and one example value of every column.

In [2]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unique_values": df.nunique(),
    "missing (isna)": df.isna().sum(),
    "example": df.iloc[0],
})
print(summary)
print()
print("Memory used:", round(df.memory_usage(deep=True).sum() / 1024), "KB")
print(df.dtypes.astype(str).value_counts().to_dict())

                    dtype  unique_values  missing (isna)           example
customerID            str           7043               0        7590-VHVEG
gender                str              2               0            Female
SeniorCitizen       int64              2               0                 0
Partner               str              2               0               Yes
Dependents            str              2               0                No
tenure              int64             73               0                 1
PhoneService          str              2               0                No
MultipleLines         str              3               0  No phone service
InternetService       str              3               0               DSL
OnlineSecurity        str              3               0                No
OnlineBackup          str              3               0               Yes
DeviceProtection      str              3               0                No
TechSupport           str

**What the code does:**
- `df.dtypes` gives the type that pandas selected for every column.
- `df.nunique()` counts the different values in each column. This is important because the number of unique values tells what the column really is (2 values means binary, 7000 values means probably an ID or a number).
- `df.isna().sum()` counts the real missing values.
- `df.iloc[0]` shows the first row, so I can see what the values look like.
- `memory_usage(deep=True)` measures the real memory, including the text inside object columns.

**Why this method was used:** The dtype alone is not enough, because pandas only guesses it when the file is read. A column can be `object` and still be a number (`TotalCharges`). So I look at the dtype together with the unique values and an example. A single `df.info()` shows dtypes and non-null counts, but not the unique values, so I built my own table.

**Before:** I only know the column names.

**Result:** Only `SeniorCitizen`, `tenure` and `MonthlyCharges` are numeric. All other columns, including `TotalCharges`, are text. Many text columns have only 2 or 3 unique values, which means they are categories and not free text.

**Effect on ML:** The dtype of a column decides how the preprocessing treats it. If a number is stored as text, the model pipeline will treat it as a category and create thousands of useless columns.

## 3. Types of Data

**Concept:** Besides the pandas dtype, every column has a logical type that comes from its meaning:

| Type | Meaning | Pandas dtype |
|---|---|---|
| Numerical | numbers that can be added and averaged | `int64`, `float64` |
| Categorical | a fixed set of labels | `category` or `object` |
| Boolean | only two values, True / False | `bool` |
| Date/Time | a point in time | `datetime64` |
| String | free text or an identifier | `object` / `str` |

**Examples:**
- Numerical: tenure in months.
- Categorical: contract type (Month-to-month, One year, Two year).
- Boolean: has partner (Yes/No).
- Date/Time: the signup date of a customer.
- String: the customer ID.

**Implementation:** I find which Telco columns belong to each logical type.

In [3]:
text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]

# a text column is boolean-like if its only values are Yes and No
yes_no_cols = [c for c in text_cols if set(df[c].dropna().unique()) == {"Yes", "No"}]
id_cols = [c for c in text_cols if df[c].nunique() == len(df)]
category_cols = [c for c in text_cols if c not in yes_no_cols + id_cols and c != "TotalCharges"]
date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]

print("Numerical (by dtype)        :", [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])])
print("Numerical stored as text    :", ["TotalCharges"])
print("Boolean-like (Yes/No text)  :", yes_no_cols)
print("Categorical (3+ labels)     :", category_cols)
print("String / identifier         :", id_cols)
print("Date/Time columns           :", date_cols if date_cols else "none")

Numerical (by dtype)        : ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Numerical stored as text    : ['TotalCharges']
Boolean-like (Yes/No text)  : ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
Categorical (3+ labels)     : ['gender', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
String / identifier         : ['customerID']
Date/Time columns           : none


**What the code does:**
- It first makes a list of all columns that are not numeric (the text columns).
- A text column is "boolean-like" when its set of unique values is exactly {Yes, No}. (The target `Churn` also matches this rule.)
- A text column where every row has a different value (unique count equals the number of rows) is an identifier.
- The remaining text columns, except `TotalCharges`, are categorical.
- It also searches the column names for "date" or "time".

**Why this method was used:** I used the data itself (the unique values) to decide the type, because the column name or the dtype can be misleading. Writing it as a rule also makes it reusable, so I can run the same code on a new dataset. I excluded `TotalCharges` by name because I already know it is a number, and I confirm this with a proper check in Section 9.

**Result:** Four yes/no features (plus the target `Churn`), eleven categorical columns, one identifier and no date column. `SeniorCitizen` is numeric by dtype, but its values are only 0 and 1, so it is also a yes/no feature.

**Effect on ML:** Each group gets a different treatment later: numerical columns are scaled, yes/no columns become 0/1, categorical columns are one-hot encoded, and the identifier is dropped.

## 4. Why Correct Data Types Are Important

**Concept:** The data type decides what pandas can do with a column: sorting, comparing, averaging, memory use and the way a model treats the column. A wrong type does not always give an error. Sometimes it gives a wrong answer without any warning, and this is the dangerous case.

**Examples:**
- Simple: as text, "100" is smaller than "20", because text is compared character by character.
- Real-world: a spreadsheet column of prices stored as text cannot be summed.
- Business: the total revenue report fails when `TotalCharges` is text.
- AI/ML: a scaler cannot work on text, and a numeric column read as a category creates thousands of one-hot columns.

**Implementation:** I show four problems that come from the wrong type of `TotalCharges`, and the memory advantage of a correct type for `Contract`.

In [4]:
total_text = df["TotalCharges"]
blank = total_text.astype(str).str.strip().eq("")
total_num = pd.to_numeric(total_text, errors="coerce")

# 1. sorting as text vs as number
print("Biggest 3 values sorted as TEXT   :", total_text[~blank].sort_values().tail(3).tolist())
print("Biggest 3 values sorted as NUMBERS:", total_num.nlargest(3).tolist())

# 2. comparison
print("\n'999.9' > '8684.8' as text:", "999.9" > "8684.8", "| 999.9 > 8684.8 as numbers:", 999.9 > 8684.8)

# 3. calculation
try:
    print("\nmean of text column:", total_text.mean())
except Exception as e:
    print("\nmean of text column fails with:", type(e).__name__)
print("mean of numeric column:", round(total_num.mean(), 2))

# 4. describe()
print("\ndescribe() of the text column:")
print(total_text.describe())
print("\ndescribe() of the numeric column:")
print(total_num.describe().round(2))

# 5. memory of text vs category
mem_text = df["Contract"].memory_usage(deep=True)
mem_cat = df["Contract"].astype("category").memory_usage(deep=True)
print("\nContract memory as text:", mem_text, "bytes | as category:", mem_cat, "bytes")

Biggest 3 values sorted as TEXT   : ['999.45', '999.8', '999.9']
Biggest 3 values sorted as NUMBERS: [8684.8, 8672.45, 8670.1]

'999.9' > '8684.8' as text: True | 999.9 > 8684.8 as numbers: False

mean of text column fails with: TypeError
mean of numeric column: 2283.3

describe() of the text column:
count     7043
unique    6531
top       20.2
freq        11
Name: TotalCharges, dtype: object

describe() of the numeric column:
count    7032.00
mean     2283.30
std      2266.77
min        18.80
25%       401.45
50%      1397.48
75%      3794.74
max      8684.80
Name: TotalCharges, dtype: float64

Contract memory as text: 136070 bytes | as category: 7230 bytes


**What the code does:**
- It copies `TotalCharges` in two versions: the original text, and a numeric version made by `to_numeric`.
- It sorts both versions and prints the largest values, to show that text sorting gives a different order.
- It compares two numbers as text and as numbers.
- It tries to calculate the mean of the text column, which fails, and then of the numeric column.
- `describe()` is run on both, because pandas gives different information for text and for numbers.
- It measures the memory of `Contract` as text and as `category`.

**Why this method was used:** Showing the problem with real values is more convincing than only writing the theory. I used `sort_values`, a comparison, `mean` and `describe` because these are the operations that an analyst uses every day, and each one breaks differently. The memory test shows that a correct type is not only about correctness: `category` stores each label once and uses small codes for the rows.

**Output meaning:** Sorted as text, the "largest" values are not the real largest amounts. As text `"999.9"` is bigger than `"8684.8"`. The mean of the text column raises an error, and `describe()` of the text only gives count, unique, top and freq, with no mean, min or max. The `category` version of `Contract` uses much less memory than the text version.

**Effect on ML:** Any feature that depends on order or size (scaling, outlier detection, correlation, transformation) gives wrong results or fails when the number is stored as text. So fixing types must be done before every other preprocessing step.

## 5. Converting Data Types with `astype()`

**Concept:** `astype()` converts a whole column to the type that I name, for example `astype("float64")`, `astype("category")` or `astype(bool)`. It works well when every value in the column can be converted. If one value cannot be converted, it raises an error, and for some conversions it gives a wrong result without any error.

I practise four conversions: Integer to Float, Text to Category, Yes/No to Boolean and Numeric to Categorical.

### 5.1 Integer to Float

**Examples:**
- Simple: `5` becomes `5.0`.
- Real-world: a count of items becomes a float when an average is needed.
- Business: a customer count becomes `NaN` when one branch has not reported.
- AI/ML: scalers and models return floats, and integer columns cannot store missing values.

In [5]:
tmp = df[["tenure"]].copy()
print("original dtype:", tmp["tenure"].dtype)

tmp["tenure_float"] = tmp["tenure"].astype("float64")
print("after astype float64:", tmp["tenure_float"].dtype, "| first values:", tmp["tenure_float"].head(3).tolist())

# why this matters: a plain integer column cannot hold a missing value
print("\nints with a missing value, pandas gives:", pd.Series([1, 2, 3, None]).dtype)
try:
    pd.Series([1, 2, 3, None], dtype="int64")
except Exception as e:
    print("forcing int64 with a missing value fails:", type(e).__name__)

original dtype: int64
after astype float64: float64 | first values: [1.0, 34.0, 2.0]

ints with a missing value, pandas gives: float64
forcing int64 with a missing value fails: TypeError


**What the code does:**
- It copies `tenure` (an integer column) and converts it to `float64` with `astype`.
- It then builds a small series `[1, 2, 3, None]`. pandas automatically makes it `float64`, because the missing value cannot be stored in an integer column.
- When I force `dtype="int64"` on the same values, pandas raises an error.

**Why this method was used:** `astype("float64")` is the right tool because every integer can be converted to a float without losing information. I included the NaN test because it explains the real reason for this conversion: standard integer columns in pandas cannot hold missing values, so a column with missing values is always float.

**Before:** `int64`, values like `1, 34, 2`.

**After:** `float64`, values like `1.0, 34.0, 2.0`. The numbers are the same.

**Why appropriate:** The conversion is safe and reversible when there are no missing values. I do not convert `tenure` in the final data, because it has no missing values and it is a whole number by meaning. I only needed to understand when it happens.

**Effect on ML:** This is why, after imputation or scaling, integer columns appear as floats. It is expected and it does not change the information.

### 5.2 Text to Category
**Examples:**
- Simple: "red", "green", "red" stored as a category with 2 labels.
- Real-world: country names in a survey.
- Business: contract types in the Telco data.
- AI/ML: pandas `category` is memory efficient, and encoders can read the labels from it.

In [6]:
cat_demo = df[["Contract", "PaymentMethod", "InternetService"]].copy()
before = cat_demo.memory_usage(deep=True).sum()

for col in cat_demo.columns:
    cat_demo[col] = cat_demo[col].astype("category")

after = cat_demo.memory_usage(deep=True).sum()
print("dtypes after:", cat_demo.dtypes.astype(str).to_dict())
print("memory before:", before, "bytes | after:", after, "bytes")
print("\nLabels of Contract:", list(cat_demo["Contract"].cat.categories))
print("Internal codes of the first rows:", cat_demo["Contract"].cat.codes.head().tolist())

dtypes after: {'Contract': 'category', 'PaymentMethod': 'category', 'InternetService': 'category'}
memory before: 423919 bytes | after: 21466 bytes

Labels of Contract: ['Month-to-month', 'One year', 'Two year']
Internal codes of the first rows: [0, 1, 0, 1, 0]


**What the code does:**
- It takes three text columns that contain only a few repeated labels.
- It converts each one to `category` with `astype("category")` and compares the memory before and after.
- `.cat.categories` shows the list of labels, and `.cat.codes` shows the small integer codes that pandas stores internally.

**Why this method was used:** These columns repeat the same 3 or 4 labels over 7043 rows. A `category` stores every label only once and keeps a small integer for each row, so it is smaller and faster than storing the full text 7043 times. It also tells everybody that this column has a fixed set of values. I did not use it for `customerID`, because it has a different value in every row, and a category would save nothing.

**Before:** `object`/`str`, one full text per row.

**After:** `category`, the same labels and much less memory.

**Why appropriate:** Nothing is lost because the labels stay the same. I have to be careful that the internal codes (0, 1, 2) are not the same as an ordering: the codes follow alphabetical order, so "Month-to-month" = 0 does not mean it is lower in any real sense. Encoding is decided in Notebook 7.

**Effect on ML:** Less memory and a clear set of categories. The codes are not features and must not be given to a model as numbers.

### 5.3 Yes/No Text to Boolean
**Examples:**
- Simple: "Yes"/"No" becomes True/False.
- Real-world: a checkbox "I agree" in a form.
- Business: does the customer have a partner?
- AI/ML: a boolean column is already a 0/1 feature for most models.

In [7]:
bool_demo = df[["Partner", "Dependents", "PhoneService", "PaperlessBilling", "SeniorCitizen"]].copy()

# WRONG: astype(bool) on text
wrong = bool_demo["Partner"].astype(bool)
print("WRONG  astype(bool) on Partner:", wrong.value_counts().to_dict())
print("actual values in Partner      :", bool_demo["Partner"].value_counts().to_dict())

# CORRECT: map the labels
bool_demo["Partner"] = bool_demo["Partner"].map({"Yes": True, "No": False})
for col in ["Dependents", "PhoneService", "PaperlessBilling"]:
    bool_demo[col] = bool_demo[col].map({"Yes": True, "No": False})

# SeniorCitizen is 0/1, so astype(bool) is correct here
bool_demo["SeniorCitizen"] = bool_demo["SeniorCitizen"].astype(bool)

print("\ndtypes after:", bool_demo.dtypes.astype(str).to_dict())
print("Missing values created by the mapping:", int(bool_demo.isna().sum().sum()))
print("Share of True per column (%):")
print((bool_demo.mean() * 100).round(1))

WRONG  astype(bool) on Partner: {True: 7043}
actual values in Partner      : {'No': 3641, 'Yes': 3402}

dtypes after: {'Partner': 'bool', 'Dependents': 'bool', 'PhoneService': 'bool', 'PaperlessBilling': 'bool', 'SeniorCitizen': 'bool'}
Missing values created by the mapping: 0
Share of True per column (%):
Partner             48.3
Dependents          30.0
PhoneService        90.3
PaperlessBilling    59.2
SeniorCitizen       16.2
dtype: float64


**What the code does:**
- First it shows a common mistake: `astype(bool)` on text. Python treats every non-empty string as True, so even "No" becomes True, and all rows become True.
- Then it converts correctly with `map({"Yes": True, "No": False})`, which says explicitly which label means which value.
- `SeniorCitizen` holds the numbers 0 and 1, so `astype(bool)` is correct there (0 is False, anything else is True).
- It counts the missing values after the mapping, to check that nothing was lost, and prints the share of True for every column.

**Why this method was used:** `map` with a dictionary is the safe way, because any value that is not in the dictionary becomes `NaN` and is visible in the check. If a hidden label like "yes " or "Y" existed, I would see it as a new missing value. `astype(bool)` does not do any check, so it hides the problem. I show the wrong version on purpose, because it is a mistake that gives no error and no warning.

**Before:** Four text columns with "Yes"/"No" and one integer column with 0/1, all different representations of the same idea.

**After:** Five `bool` columns with True/False, and no new missing values.

**Why appropriate:** These five columns have exactly two possible values with a clear meaning, so boolean is their natural type. The means show the share of customers that are True, for example the share of senior citizens.

**Effect on ML:** All binary features now look the same, which solves Sprint 4 problem 2. Models treat True/False as 1/0, so the final 0/1 encoding in Notebook 7 is simple.

### 5.4 Numeric to Categorical
**Examples:**
- Simple: the codes 1, 2, 3 for small, medium, large are labels, not quantities.
- Real-world: a zip code or a phone number is stored as a number but cannot be averaged.
- Business: a customer is grouped into a tenure band like "0-12 months".
- AI/ML: a model would treat zip code 90000 as bigger than 10000, which is meaningless, so it must be a category.

In [8]:
num_cat = df[["tenure", "SeniorCitizen"]].copy()

# case 1: numeric code that is really a label
num_cat["SeniorCitizen_label"] = num_cat["SeniorCitizen"].map({0: "No", 1: "Yes"}).astype("category")

# case 2: a real number grouped into bands (demo only)
num_cat["tenure_group"] = pd.cut(num_cat["tenure"], bins=[-1, 12, 24, 48, 72],
                                 labels=["0-12", "13-24", "25-48", "49-72"])

print(num_cat.dtypes.astype(str).to_dict())
print("\nSeniorCitizen_label counts:", num_cat["SeniorCitizen_label"].value_counts().to_dict())
print("\ntenure_group counts:")
print(num_cat["tenure_group"].value_counts().sort_index())

{'tenure': 'int64', 'SeniorCitizen': 'int64', 'SeniorCitizen_label': 'category', 'tenure_group': 'category'}

SeniorCitizen_label counts: {'No': 5901, 'Yes': 1142}

tenure_group counts:
tenure_group
0-12     2186
13-24    1024
25-48    1594
49-72    2239
Name: count, dtype: int64


**What the code does:**
- Case 1: it replaces the codes 0 and 1 of `SeniorCitizen` with readable labels using `map`, and stores the result as a category.
- Case 2: it uses `pd.cut` to divide `tenure` (a real number) into four bands with names. The bins are `-1` to `12`, `12` to `24`, `24` to `48` and `48` to `72`, and the `-1` is there so that tenure 0 falls into the first band.
- It prints the dtypes and the number of customers in each group.

**Why this method was used:** The two cases are different. In case 1 the number has no numeric meaning, so the correct step is to rename the codes, and `map` does exactly that. In case 2 the number is meaningful and I only group it. `pd.cut` is used because I choose the band edges myself, based on business logic. `qcut` would make groups of equal size, which hides the real business periods.

**Before:** `int64` columns, where 0/1 or 1 to 72 look like quantities.

**After:** Two `category` columns with readable labels.

**Why appropriate:** I use this only when a number is really a label. I do not convert `tenure` itself, because the exact number carries information that the bands lose. In the final dataset `SeniorCitizen` stays boolean (decision of 5.3), and `tenure_group` is only a demo. Creating new features from bands belongs to the Feature Engineering sprint.

**Effect on ML:** If a code is treated as a number, a model can invent an order and distance that do not exist. Converting it to a category stops that. Grouping a number loses detail, so I do it only when there is a reason.

## 6. Converting Text to Numbers with `to_numeric()`

**Concept:** `pd.to_numeric()` converts text to numbers, and it has an `errors` option that decides what happens to a value that cannot be converted: `errors="raise"` (default) stops with an error, and `errors="coerce"` replaces the bad value with `NaN`. It also has a `downcast` option that chooses the smallest numeric type that can hold the values.

**Examples:**
- Simple: `"25"` becomes `25`.
- Real-world: prices copied from a website come as text.
- Business: `TotalCharges` is the amount that a customer has paid.
- AI/ML: a numeric feature stored as text cannot be scaled.

**Implementation:** First I look for the values that cannot be converted, then I convert `TotalCharges` with both options.

In [9]:
raw_total = df["TotalCharges"]

# step 1: what happens with errors="raise"
try:
    pd.to_numeric(raw_total, errors="raise")
except Exception as e:
    print("errors='raise' ->", type(e).__name__, ":", str(e)[:70])

# step 2: find the bad values BEFORE converting
converted = pd.to_numeric(raw_total, errors="coerce")
bad = raw_total[converted.isna()]
print("\nValues that cannot be converted:", len(bad))
print("Unique bad values:", bad.unique().tolist())
print("tenure of those customers:", df.loc[bad.index, "tenure"].unique().tolist())

# step 3: convert with errors="coerce"
total_float = converted
print("\nAfter coerce -> dtype:", total_float.dtype, "| NaN count:", int(total_float.isna().sum()))

# step 4: downcast to save memory
tenure_small = pd.to_numeric(df["tenure"], downcast="integer")
print("\ntenure dtype", df["tenure"].dtype, "->", tenure_small.dtype,
      "| memory:", df["tenure"].memory_usage(), "->", tenure_small.memory_usage(), "bytes")

errors='raise' -> ValueError : Unable to parse string " " at position 488

Values that cannot be converted: 11
Unique bad values: [' ']
tenure of those customers: [0]

After coerce -> dtype: float64 | NaN count: 11

tenure dtype int64 -> int8 | memory: 56476 -> 7175 bytes


**What the code does:**
- Step 1 calls `to_numeric` with `errors="raise"`. This stops at the first value that cannot be converted and prints the error.
- Step 2 converts with `errors="coerce"` and then compares: the rows that became `NaN` after the conversion are the bad values. Then it prints the bad values and the `tenure` of those customers.
- Step 3 keeps the converted column, and shows its dtype and the number of `NaN`.
- Step 4 uses `downcast="integer"` on `tenure`, to ask pandas for the smallest integer type that can store the values (the maximum tenure is 72, so `int8` is enough).

**Why this method was used:** I use `errors="raise"` first to learn that a problem exists, because if I use `coerce` straight away, the bad values disappear and I never see them. Then I use `coerce` because I want to keep the 7032 valid values and mark only the 11 bad ones as missing. `astype(float)` is not a good option here, because it fails on the first blank string and does not show which values are bad. Looking at the bad values before converting is important, because I found out that all of them are blank spaces of customers with tenure 0. That is a business reason and not random dirt.

**Before:** `TotalCharges` is text with 11 blank strings, so `isna()` finds nothing.

**After:** `float64`, with 11 real `NaN` values that pandas can see and count.

**Why appropriate:** The conversion does not decide what to do with the missing values. It only turns hidden missing values into visible ones. What number to put there is decided in Notebook 3, with a business reason. In this notebook I keep them as `NaN`. Downcasting `tenure` is shown only to explain `downcast`. I do not apply it, because 7043 rows are small and the memory saving is not needed.

**Effect on ML:** `TotalCharges` can now be used for scaling, outlier detection and transformation, and the 11 missing values can be handled with a documented method. The hidden missing values are now counted correctly, which fixes Sprint 4 problem 1.

## 7. Converting Text to Dates with `to_datetime()`

**Concept:** `pd.to_datetime()` converts text to a real date type (`datetime64`). With a real date, I can calculate differences, sort by time and take parts such as year, month or weekday. The `format` option tells pandas how the text is written (for example `%d-%m-%Y` is day-month-year), and `errors="coerce"` makes an invalid date `NaT` (not a time), which is the missing value of dates.

**Examples:**
- Simple: `"15-01-2023"` becomes the date 15 January 2023.
- Real-world: `03/04/2024` is 3 April in India and 4 March in the USA.
- Business: the number of days since a customer signed up.
- AI/ML: models cannot read a date text, but they can use features like month or days since signup.

**Implementation:** The Telco dataset has no date column, so I use a small demo table that I made. It has the usual problems of real dates: different formats, an invalid date and text instead of a date.

In [10]:
# DEMO data (not from the Telco dataset)
dates = pd.DataFrame({
    "customer": ["A", "B", "C", "D", "E", "F", "G"],
    "signup": ["15-01-2023", "03-04-2024", "31-12-2022", "2023/05/09", "not available", "", "29-02-2023"],
})
print("dtype:", dates["signup"].dtype)

# step 1: parse with one explicit format
parsed = pd.to_datetime(dates["signup"], format="%d-%m-%Y", errors="coerce")
print("\nParsed with ONE format -> NaT count:", int(parsed.isna().sum()))

# step 2: second format for the values that failed, combine both
parsed2 = pd.to_datetime(dates["signup"], format="%Y/%m/%d", errors="coerce")
dates["signup_date"] = parsed.fillna(parsed2)
print(dates[["signup", "signup_date"]])

# step 3: the day/month trap
print("\n'03-04-2024' as day-month:", pd.to_datetime("03-04-2024", format="%d-%m-%Y").date(),
      "| as month-day:", pd.to_datetime("03-04-2024", format="%m-%d-%Y").date())

# step 4: use the date
reference = pd.Timestamp("2024-06-30")
dates["year"] = dates["signup_date"].dt.year
dates["month"] = dates["signup_date"].dt.month_name()
dates["days_since_signup"] = (reference - dates["signup_date"]).dt.days
print()
print(dates[["customer", "signup_date", "year", "month", "days_since_signup"]])
print("\nfinal dtype:", dates["signup_date"].dtype)

dtype: str

Parsed with ONE format -> NaT count: 4
          signup signup_date
0     15-01-2023  2023-01-15
1     03-04-2024  2024-04-03
2     31-12-2022  2022-12-31
3     2023/05/09  2023-05-09
4  not available         NaT
5                        NaT
6     29-02-2023         NaT

'03-04-2024' as day-month: 2024-04-03 | as month-day: 2024-03-04

  customer signup_date    year     month  days_since_signup
0        A  2023-01-15  2023.0   January              532.0
1        B  2024-04-03  2024.0     April               88.0
2        C  2022-12-31  2022.0  December              547.0
3        D  2023-05-09  2023.0       May              418.0
4        E         NaT     NaN       NaN                NaN
5        F         NaT     NaN       NaN                NaN
6        G         NaT     NaN       NaN                NaN

final dtype: datetime64[us]


**What the code does:**
- It creates a table of 7 signup dates written in different ways: normal dates, one date in another format (`2023/05/09`), a text ("not available"), an empty cell and an impossible date (29 February 2023 does not exist).
- Step 1 parses with the format `%d-%m-%Y` and `errors="coerce"`, so all values that do not match become `NaT`.
- Step 2 parses again with the second format, and `fillna` takes the parsed value from the second result only where the first one failed.
- Step 3 shows that `03-04-2024` gives two different dates depending on the format.
- Step 4 uses the `.dt` accessor to take year and month name, and subtracts a reference date to get the number of days since signup.

**Why this method was used:** I give an explicit `format` and do not let pandas guess. If pandas guesses, it can read the day and the month in the wrong order without any warning, and the whole column can be wrong. With an explicit format a value that does not match becomes `NaT`, so I can count and check it. I used two formats and combined them, because the demo column mixes formats, and this keeps each format strict. `errors="coerce"` is used so that one bad date does not stop the whole conversion. The impossible date 29-02-2023 is caught automatically, which shows that `to_datetime` also validates the date.

**Before:** Text column, so no subtraction, sorting by time or month extraction is possible.

**After:** `datetime64` column. The valid dates are real dates, and the three bad values ("not available", empty, 29-02-2023) are `NaT`.

**Why appropriate:** The `NaT` values are not hidden or invented. They are visible and I can decide later what to do with them. The reference date is fixed, so the result can be repeated.

**Effect on ML:** A model cannot use a raw date, but features such as year, month, weekday and days since signup are useful. For time based data, the correct date type is also needed for temporal splitting (Notebook 12) to avoid using the future to predict the past.

## 8. Handling Invalid Conversions

**Concept:** A conversion is invalid when a value cannot become the target type, for example "thirty" as a number, or "N/A" as a date. There are three ways to handle it: stop with an error (`raise`), turn the value into a missing value (`coerce`), or clean the text first so that it can be converted. The important rule is never to coerce blindly: first look at what will be lost, then decide.

**Examples:**
- Simple: "N/A" cannot become a number.
- Real-world: "$1,200" has a symbol and a comma that block the conversion.
- Business: a price column from different suppliers with different formats.
- AI/ML: if many values silently become `NaN`, the model trains on much less data than I think.

**Implementation:** I use a messy demo series (my own example), and I write a small function that reports what a conversion changes.

In [11]:
# DEMO data
messy = pd.Series(["25", "31", "thirty", "N/A", "1,200", "$45", " 60 ", ""], name="amount")

def check_conversion(series):
    # converts with coerce and reports which values were lost
    converted = pd.to_numeric(series, errors="coerce")
    newly_missing = converted.isna() & series.notna()
    print("Values:", len(series), "| converted:", int(converted.notna().sum()), "| lost (became NaN):", int(newly_missing.sum()))
    print("Lost values:", series[newly_missing].tolist())
    return converted

print("--- direct conversion")
direct = check_conversion(messy)

print("\n--- after cleaning the text first")
cleaned = messy.str.strip().str.replace(r"[$,]", "", regex=True)
fixed = check_conversion(cleaned)

report = pd.DataFrame({"original": messy, "cleaned": cleaned, "direct": direct, "after_cleaning": fixed})
print()
print(report)

print("\n--- the same check on the real TotalCharges column")
_ = check_conversion(df["TotalCharges"])

--- direct conversion
Values: 8 | converted: 3 | lost (became NaN): 5
Lost values: ['thirty', 'N/A', '1,200', '$45', '']

--- after cleaning the text first
Values: 8 | converted: 5 | lost (became NaN): 3
Lost values: ['thirty', 'N/A', '']

  original cleaned  direct  after_cleaning
0       25      25    25.0            25.0
1       31      31    31.0            31.0
2   thirty  thirty     NaN             NaN
3      N/A     N/A     NaN             NaN
4    1,200    1200     NaN          1200.0
5      $45      45     NaN            45.0
6      60       60    60.0            60.0
7                      NaN             NaN

--- the same check on the real TotalCharges column
Values: 7043 | converted: 7032 | lost (became NaN): 11
Lost values: [' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ']


**What the code does:**
- It creates a demo series with typical problems: a word ("thirty"), a missing marker ("N/A"), a comma in a number ("1,200"), a currency symbol ("$45"), spaces and an empty string.
- The function `check_conversion` converts with `errors="coerce"`, then compares: a value that was not missing before, but is `NaN` after, was lost by the conversion. It prints how many were lost and which ones.
- First it converts directly, then it removes spaces, `$` and `,` with `str.strip()` and `str.replace()` and converts again.
- At the end the same function runs on the real `TotalCharges`.

**Why this method was used:** Plain `coerce` hides the loss, because the result is a column with some `NaN` and I do not know what was lost. The function makes the loss visible every time, as a count and a list. Cleaning before converting saves the values that only have a formatting problem ("1,200", "$45", " 60 "), because these are real numbers. Values like "thirty" and "N/A" stay missing, because there is no safe rule to turn them into a number, and inventing one would be a guess. I use a regular expression `[$,]` so that one line removes both symbols.

**Before:** Direct conversion loses many values that were good numbers with bad formatting.

**After:** Only the values that really are not numbers ("thirty", "N/A", the empty value) are lost.

**Why appropriate:** The rule is: save what can be saved with a clear rule, mark what cannot as missing, and always report how many values were lost. For `TotalCharges` the report shows 11 lost values, the same 11 that I already know and understand.

**Effect on ML:** If a conversion silently removes 20% of a column, the model learns from a different dataset than I believe. The loss report is a protection, and I can run the same function in the final pipeline as a validation step.

## 9. Detecting Incorrect Data Types

**Concept:** A column has an incorrect type when its dtype does not match its meaning. The common cases are: numbers stored as text, yes/no values stored as text, a binary or code column stored as a number, dates stored as text, and an identifier stored as a normal feature. I can detect them by comparing the dtype with the real content of the column.

**Examples:**
- Simple: a column "age" with dtype `object`.
- Real-world: a column "joined" with text like "2023-01-15".
- Business: `TotalCharges` is a money value but it is text.
- AI/ML: a wrongly typed column is handled with the wrong preprocessing, so the model gets a wrong feature.

**Implementation:** I write a detector that checks every column by its content, and I run it on the raw data.

In [12]:
def detect_wrong_types(data, numeric_share=0.9, low_cardinality=0.05):
    findings = []
    for col in data.columns:
        s = data[col].dropna()
        is_num = pd.api.types.is_numeric_dtype(data[col]) and not pd.api.types.is_bool_dtype(data[col])

        if is_num:
            n = s.nunique()
            if n == 2:
                findings.append((col, str(data[col].dtype), "number with only 2 values", "boolean"))
            elif n <= 10:
                findings.append((col, str(data[col].dtype), "number with few values (" + str(n) + ")", "check: category?"))
        elif pd.api.types.is_string_dtype(data[col]) or data[col].dtype == "object":
            text = s.astype(str).str.strip()
            looks_numeric = pd.to_numeric(text, errors="coerce").notna().mean()
            if looks_numeric >= numeric_share:
                findings.append((col, str(data[col].dtype), str(round(looks_numeric * 100, 1)) + "% of values look like numbers", "numeric (to_numeric)"))
            elif set(text.unique()) == {"Yes", "No"}:
                findings.append((col, str(data[col].dtype), "only Yes / No", "boolean"))
            elif text.nunique() == len(text):
                findings.append((col, str(data[col].dtype), "different value in every row", "identifier: drop"))
            elif text.nunique() / len(text) < low_cardinality and not isinstance(data[col].dtype, pd.CategoricalDtype):
                findings.append((col, str(data[col].dtype), "few repeated labels (" + str(text.nunique()) + ")", "category"))
        if any(k in col.lower() for k in ["date", "time"]) and col not in [f[0] for f in findings]:
            findings.append((col, str(data[col].dtype), "name suggests a date", "datetime (to_datetime)"))

    return pd.DataFrame(findings, columns=["column", "current_dtype", "reason", "suggested_type"])

detected = detect_wrong_types(df)
print("Columns flagged:", len(detected), "of", df.shape[1])
detected

Columns flagged: 19 of 21


,column,current_dtype,reason,suggested_type
0,customerID,str,different value in every row,identifier: drop
1,gender,str,few repeated labels (2),category
2,SeniorCitizen,int64,number with only 2 values,boolean
3,Partner,str,only Yes / No,boolean
4,Dependents,str,only Yes / No,boolean
5,PhoneService,str,only Yes / No,boolean
6,MultipleLines,str,few repeated labels (3),category
7,InternetService,str,few repeated labels (3),category
8,OnlineSecurity,str,few repeated labels (3),category
9,OnlineBackup,str,few repeated labels (3),category


**What the code does:** The function goes through every column and decides by content, not by name:
- Numeric columns: if there are only 2 unique values, it suggests boolean. With 10 or fewer values, it suggests that the column may be a category.
- Text columns: if at least 90% of the values can be converted to numbers, it says the column is a number stored as text. If the values are only Yes and No, it suggests boolean. If every row has a different value, it is an identifier. If there are very few repeated labels (less than 5% of the rows), it suggests category.
- If a column is already a `category`, it is not flagged again.
- It also flags names that contain "date" or "time".
- It returns a table with the column, the current dtype, the reason and the suggested type.

**Why this method was used:** I wrote one function instead of checking 21 columns by eye, so it is repeatable and can be used on any dataset and inside the final workflow. The thresholds (90% and 5%) are parameters that I can change. I used 90% and not 100% for the numeric check, because the whole point is to find columns like `TotalCharges` where a few bad values hide the fact that the column is a number. If I asked for 100%, that column would not be flagged. The function only suggests a type, and it does not change anything, because a human must confirm each change.

**Output meaning:** `TotalCharges` is detected as a number stored as text (almost all values look like numbers). The yes/no columns are detected as boolean, `SeniorCitizen` is detected as a number with 2 values, `customerID` as an identifier and the other text columns as categories.

**Effect on ML:** The detector finds exactly the problems from the Sprint 4 list, so the fixes in the next section are based on evidence. The same function can be used as a validation step to protect the pipeline against new wrong types.

## 10. Fixing the Data Types of the Telco Dataset

**Concept:** Now I apply the decisions as one function, so that the same fixes can be used again in the final pipeline (Notebook 16) and the raw data stays unchanged.

| Column(s) | Decision | Reason |
|---|---|---|
| `TotalCharges` | `to_numeric(errors="coerce")` | it is a number; the 11 blanks become visible `NaN` |
| `SeniorCitizen` | 0/1 to `bool` | it is a yes/no feature |
| `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling` | "Yes"/"No" to `bool` | only two values |
| `gender`, `MultipleLines`, `InternetService`, 6 add-on columns, `Contract`, `PaymentMethod`, `Churn` | `category` | a fixed set of labels |
| `customerID` | keep as text | an identifier, will be dropped later |
| `tenure`, `MonthlyCharges` | no change | already correct |

**Examples:**
- Simple: a column with 2 labels becomes boolean.
- Real-world: a form with checkboxes and drop-down lists.
- Business: every customer attribute has one clear type.
- AI/ML: the pipeline can now select columns by dtype (numeric, bool, category).

**Implementation:**

In [13]:
def fix_data_types(data):
    out = data.copy()                                   # raw data is never changed

    # text -> number (bad values become NaN, handled in Notebook 3)
    out["TotalCharges"] = pd.to_numeric(out["TotalCharges"], errors="coerce")

    # yes/no features -> boolean
    out["SeniorCitizen"] = out["SeniorCitizen"].astype(bool)
    for col in ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]:
        out[col] = out[col].map({"Yes": True, "No": False})

    # labels -> category
    category_columns = ["gender", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
                        "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
                        "Contract", "PaymentMethod", "Churn"]
    for col in category_columns:
        out[col] = out[col].astype("category")

    return out

df_clean = fix_data_types(df)

# check: nothing was lost by the conversion, raw data untouched
print("Raw TotalCharges dtype still:", df["TotalCharges"].dtype)
print("New missing values:", df_clean.isna().sum()[df_clean.isna().sum() > 0].to_dict())
print("Rows:", len(df_clean), "| Columns:", df_clean.shape[1])

before_after = pd.DataFrame({
    "before": df.dtypes.astype(str),
    "after": df_clean.dtypes.astype(str),
})
before_after["changed"] = before_after["before"] != before_after["after"]
print()
print(before_after)

print("\nDtype counts before:", df.dtypes.astype(str).value_counts().to_dict())
print("Dtype counts after :", df_clean.dtypes.astype(str).value_counts().to_dict())
print("Memory before:", round(df.memory_usage(deep=True).sum() / 1024), "KB | after:",
      round(df_clean.memory_usage(deep=True).sum() / 1024), "KB")

# the detector should now find much less
print("\nDetector on the fixed data:")
detect_wrong_types(df_clean)

Raw TotalCharges dtype still: str
New missing values: {'TotalCharges': 11}
Rows: 7043 | Columns: 21

                   before     after  changed
customerID            str       str    False
gender                str  category     True
SeniorCitizen       int64      bool     True
Partner               str      bool     True
Dependents            str      bool     True
tenure              int64     int64    False
PhoneService          str      bool     True
MultipleLines         str  category     True
InternetService       str  category     True
OnlineSecurity        str  category     True
OnlineBackup          str  category     True
DeviceProtection      str  category     True
TechSupport           str  category     True
StreamingTV           str  category     True
StreamingMovies       str  category     True
Contract              str  category     True
PaperlessBilling      str      bool     True
PaymentMethod         str  category     True
MonthlyCharges    float64   float64    False

,column,current_dtype,reason,suggested_type
0,customerID,str,different value in every row,identifier: drop
1,Churn,category,only Yes / No,boolean


**What the code does:**
- `fix_data_types` makes a copy of the data and applies the decisions of the table: `TotalCharges` to float, the yes/no features to boolean, and the label columns to category.
- It returns the new dataframe `df_clean`.
- After that, the checks print the dtype of the raw column (still the old one), the number of new missing values, the shape, a before/after table of all dtypes with a flag for the changed ones, the counts of dtypes, and the memory.
- The detector from Section 9 is run again on the fixed data. Only `customerID` (dropped later) and `Churn` (the target, encoded in Notebook 7) are still flagged, and I keep both on purpose.

**Why this method was used:** A function is better than separate lines, because I can run the same cleaning again on the training and test data, and on new data, in exactly the same way. Working on a copy keeps `df` untouched for the before/after analysis in Notebook 15. The checks after the conversion are important because a conversion can fail silently (for example a wrong `map` creates `NaN`). I compare the missing values, the shape and the dtypes, and I run the detector as a second opinion. These conversions are fixed rules, they do not learn anything from the data, so doing them before the train/test split is safe and does not cause leakage.

**Before:** 18 text columns, 2 integer columns and 1 float column, with `TotalCharges` as text and hidden blanks.

**After:** Float, integer, boolean and category columns. `TotalCharges` is a number with 11 visible `NaN`, and these are the only missing values in the data. The number of rows and columns is the same and the memory is much lower.

**Why appropriate:** Each decision follows the real meaning of the column, and I did not change the values, only the way they are stored. I kept `Churn` as a category, because the target encoding is decided in Notebook 7. I kept `customerID` as text and `tenure` as integer, because converting them gives no benefit.

**Effect on ML:** The data is ready for the next steps. The 11 missing values in `TotalCharges` are now visible for Notebook 3, and the pipeline can select columns by dtype: numbers for scaling, boolean and category for encoding.

In [14]:
df_clean.to_csv("telco_step2_dtypes_fixed.csv", index=False)

# update the issue log from Notebook 1 (if it exists)
if os.path.exists("data_quality_issue_log.csv"):
    log = pd.read_csv("data_quality_issue_log.csv")
    log.loc[log["id"] == 1, "status"] = "done (Notebook 2)"
    log.to_csv("data_quality_issue_log.csv", index=False)
    print(log[["id", "column", "problem", "status"]].head(3))
else:
    print("data_quality_issue_log.csv not found (run Notebook 1 first)")

   id        column                   problem             status
0   1  TotalCharges            stored as text  done (Notebook 2)
1   2  TotalCharges  11 hidden missing values            pending
2   3      all rows           duplicate check           verified


**What the code does:** It saves the fixed dataset and changes the status of issue 1 (`TotalCharges` stored as text) to done in the issue log from Notebook 1.

**Why this method was used:** A saved file and an updated log keep the sprint connected: each notebook can show which problem it solved. A CSV does not remember dtypes, so later notebooks load the original CSV and call `fix_data_types` again, instead of depending on the saved file.

## 11. Documentation of Decisions

**Decision 1 - TotalCharges: text to number**

| Item | Documentation |
|---|---|
| Problem | `TotalCharges` is stored as text and `isna()` shows no missing values |
| Analysis | 11 cells contain a blank space, all for customers with tenure 0, and one blank is enough to make pandas read the whole column as text |
| Technique selected | `pd.to_numeric(errors="coerce")` |
| Reason | `astype(float)` fails on the first blank, `raise` stops everything, and `coerce` keeps the 7032 valid values and marks only the 11 bad ones as missing |
| Implementation | Section 6 and `fix_data_types` in Section 10 |
| Result | dtype changed from text to `float64`, with 11 visible `NaN` |
| Impact | the column can be scaled and transformed, and the missing values can be handled in Notebook 3 |

**Decision 2 - Yes/No features: text and 0/1 to boolean**

| Item | Documentation |
|---|---|
| Problem | Binary features are stored in two ways: "Yes"/"No" text and 0/1 integers (`SeniorCitizen`) |
| Analysis | The columns were exported with different rules |
| Technique selected | `map({"Yes": True, "No": False})` for text and `astype(bool)` for 0/1 |
| Reason | `map` shows unknown labels as `NaN`, while `astype(bool)` on text would turn "No" into True |
| Implementation | Section 5.3 and `fix_data_types` |
| Result | 5 `bool` columns and no new missing values |
| Impact | all binary features have one format, so the 0/1 encoding in Notebook 7 is simple |

**Decision 3 - Label columns: text to category**

| Item | Documentation |
|---|---|
| Problem | Columns with a few repeated labels are stored as text |
| Analysis | pandas reads every text column as `object` and does not know that the labels repeat |
| Technique selected | `astype("category")` |
| Reason | Less memory, an explicit set of labels, and a clear signal for the encoder |
| Implementation | Section 5.2 and `fix_data_types` |
| Result | 12 `category` columns, with the same labels |
| Impact | the pipeline can select categorical columns by dtype, and the internal codes are not used as features |

# Summary

| Topic | Key idea |
|---|---|
| Identifying types | look at dtype, unique values and examples together |
| Logical types | numerical, categorical, boolean, date/time, string |
| Why types matter | wrong types give wrong sorting, failed calculations and wrong model input |
| `astype()` | converts a whole column, but it is unsafe on text (`astype(bool)` trap) |
| `to_numeric()` | `raise` shows the problem, `coerce` keeps valid values and marks bad ones |
| `to_datetime()` | use an explicit `format`, bad dates become `NaT` |
| Invalid conversions | check what is lost, clean formatting first, never coerce blindly |
| Detecting wrong types | compare the dtype with the real content of the column |

**Key findings on the Telco data:**
- `TotalCharges` is a number stored as text, because of 11 blank strings (customers with tenure 0).
- Yes/No features were stored in two ways, and now all five are boolean.
- Twelve label columns are now `category`.
- The only missing values after the conversion are the 11 `NaN` in `TotalCharges`, and they are the input of Notebook 3.

**What I learned:** A wrong data type often does not give an error, only a wrong answer, like `astype(bool)` on "No" or sorting numbers as text. So I should check the result of every conversion, and look at the bad values before I coerce them.

**Next:** Notebook 3 - Missing Value Handling.